# Perbandingan MLP dengan 3 Fungsi Aktivasi
**Dataset:** Iris (Kaggle - `Iris.csv`)

**Fungsi Aktivasi yang Dibandingkan:** Sigmoid | Tanh | ReLU

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')
print('Library berhasil diimport!')

Library berhasil diimport!


## 2. Load Dataset Iris (Kaggle)

In [2]:
# Load dataset dari file CSV
df = pd.read_csv('Iris.csv')
print('Shape dataset:', df.shape)
print()
print('Jumlah data per kelas:')
print(df['Species'].value_counts())
print()
print('5 data pertama:')
df.head()

Shape dataset: (150, 6)

Jumlah data per kelas:
Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

5 data pertama:


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 3. Preprocessing Data

In [3]:
# Hapus kolom Id (tidak relevan)
df = df.drop(columns=['Id'])

# Pisahkan fitur dan label
X = df[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']].values
y_raw = df['Species'].values

# Encode label ke angka
le = LabelEncoder()
y = le.fit_transform(y_raw)
print('Kelas yang ditemukan:', le.classes_)

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisasi dengan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Data latih : {X_train_scaled.shape[0]} sampel')
print(f'Data uji   : {X_test_scaled.shape[0]} sampel')

Kelas yang ditemukan: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']
Data latih : 120 sampel
Data uji   : 30 sampel


## 4. Training MLP dengan 3 Fungsi Aktivasi

In [4]:
activation_functions = ['logistic', 'tanh', 'relu']
activation_labels    = {'logistic': 'Sigmoid', 'tanh': 'Tanh', 'relu': 'ReLU'}
results = {}

for act in activation_functions:
    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation=act,
        solver='adam',
        max_iter=1000,
        random_state=42,
        learning_rate_init=0.001
    )
    mlp.fit(X_train_scaled, y_train)
    y_pred = mlp.predict(X_test_scaled)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec  = recall_score(y_test, y_pred, average='macro')
    f1   = f1_score(y_test, y_pred, average='macro')

    results[act] = {
        'Fungsi Aktivasi'    : activation_labels[act],
        'Accuracy (%)'       : round(acc  * 100, 2),
        'Precision (macro)'  : round(prec * 100, 2),
        'Recall (macro)'     : round(rec  * 100, 2),
        'F1-Score (macro)'   : round(f1   * 100, 2),
        'Jumlah Iterasi'     : mlp.n_iter_
    }
    print(f"[{activation_labels[act]:7s}] Accuracy: {acc*100:.2f}%  "
          f"F1: {f1*100:.2f}%  Iterasi: {mlp.n_iter_}")

print('Training selesai!')

[Sigmoid] Accuracy: 96.67%  F1: 96.66%  Iterasi: 585
[Tanh   ] Accuracy: 96.67%  F1: 96.66%  Iterasi: 312


[ReLU   ] Accuracy: 96.67%  F1: 96.66%  Iterasi: 303
Training selesai!


## 5. Tabel Perbandingan Performa

In [5]:
df_results = pd.DataFrame(list(results.values()))
print('=' * 80)
print(' TABEL PERBANDINGAN PERFORMA MLP - DATASET IRIS (KAGGLE) '.center(80, '='))
print('=' * 80)
print(df_results.to_string(index=False))
print('=' * 80)
df_results

=========== TABEL PERBANDINGAN PERFORMA MLP - DATASET IRIS (KAGGLE) ============
Fungsi Aktivasi  Accuracy (%)  Precision (macro)  Recall (macro)  F1-Score (macro)  Jumlah Iterasi
        Sigmoid         96.67              96.97           96.67             96.66             585
           Tanh         96.67              96.97           96.67             96.66             312
           ReLU         96.67              96.97           96.67             96.66             303


,Fungsi Aktivasi,Accuracy (%),Precision (macro),Recall (macro),F1-Score (macro),Jumlah Iterasi
0,Sigmoid,96.67,96.97,96.67,96.66,585
1,Tanh,96.67,96.97,96.67,96.66,312
2,ReLU,96.67,96.97,96.67,96.66,303


## 6. Classification Report per Fungsi Aktivasi

In [6]:
for act in activation_functions:
    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32), activation=act,
        solver='adam', max_iter=1000, random_state=42
    )
    mlp.fit(X_train_scaled, y_train)
    y_pred = mlp.predict(X_test_scaled)
    print(f"{'='*58}")
    print(f" Classification Report - {activation_labels[act]} ".center(58, '='))
    print(f"{'='*58}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

============ Classification Report - Sigmoid =============
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30

============== Classification Report - Tanh ==============


                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30



============== Classification Report - ReLU ==============
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30



## 7. Catatan & Kesimpulan

### Perbandingan Fungsi Aktivasi

| Fungsi Aktivasi | Kelebihan | Kekurangan |
|---|---|---|
| **Sigmoid** | Output probabilistik (0–1), mudah diinterpretasi | Rentan *vanishing gradient*, konvergensi lambat |
| **Tanh** | Zero-centered, konvergensi lebih cepat dari Sigmoid | Masih bisa mengalami *vanishing gradient* |
| **ReLU** | Komputasi cepat, menghindari *vanishing gradient* | Bisa *dying ReLU* jika learning rate terlalu besar |

### Analisis Hasil

Dataset **Iris (Kaggle)** merupakan dataset kecil (150 sampel) yang bersifat **linearly separable**, sehingga ketiga fungsi aktivasi menghasilkan akurasi yang tinggi dan hampir sama. Perbedaan utama terletak pada **kecepatan konvergensi (jumlah iterasi)**:

- **ReLU** → konvergensi paling cepat (iterasi paling sedikit)
- **Tanh** → hampir sama cepat dengan ReLU
- **Sigmoid** → paling lambat konvergen akibat efek *vanishing gradient*

### Kesimpulan

> **Fungsi aktivasi terbaik untuk Dataset Iris adalah `ReLU`**, karena menghasilkan konvergensi tercepat dengan akurasi yang sama tingginya dibanding Tanh dan Sigmoid. Untuk dataset kecil seperti Iris, `Tanh` juga merupakan pilihan yang sangat baik karena sifatnya yang zero-centered membantu proses pelatihan lebih stabil.

> Sigmoid **tidak direkomendasikan** untuk jaringan yang lebih dalam karena rentan terhadap masalah *vanishing gradient*, meskipun pada dataset Iris yang sederhana masih mampu mencapai akurasi tinggi.

---
**Nama  :** [Isi Nama Kamu]

**NIM   :** [Isi NIM Kamu]

**Kelas :** [Isi Kelas Kamu]

**Mata Kuliah :** Machine Learning / Deep Learning